# Compare Fine-Tuned LoRA Models (Colab Wrapper)

This notebook is a thin wrapper around `compare_lora_models.py`.

Run it after the TinyLlama and Llama 3.2 LoRA checkpoints both exist.
It keeps the Python script as the source of truth, while the notebook handles:
- mounting Google Drive
- entering the synced repo root
- installing dependencies
- verifying checkpoints and benchmark data exist
- launching the comparison script in a fresh Python process
- confirming that the output CSV and plot were written


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import os
from pathlib import Path

# Update this to your Drive-synced repo path before running.
REPO_ROOT = Path("/content/drive/MyDrive/UVM/Deep Learning/Final Project/repo-mirror/deep-learning-final-project")
if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repo root not found: {REPO_ROOT}")

os.chdir(REPO_ROOT)
print(f"Working directory: {Path.cwd()}")


In [ ]:
%pip install -r requirements.txt
%pip install sentence-transformers transformers datasets accelerate peft trl scikit-learn matplotlib beautifulsoup4


In [ ]:
from pathlib import Path

required_paths = [
    Path("compare_lora_models.py"),
    Path("compare_retrievers.py"),
    Path("chat_prompting.py"),
    Path("WildGraphBench"),
    Path("all_embeddings"),
    Path("tinyllama-lora-mcu"),
    Path("llama32-3b-lora-mcu"),
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required files for model comparison:\n- " + "\n- ".join(missing)
    )

for path in required_paths:
    print(f"Found: {path}")


In [ ]:
DOMAIN = "culture"
TOPIC = "Marvel Cinematic Universe"
WILDBENCH_ROOT = "WildGraphBench"
EMBEDDER_PATH = "./all_embeddings"
EMBEDDER_LABEL = "wildgraph"

K = 3
LIMIT = 50
CHUNK_SIZE = 300
WORD_COVERAGE_THRESHOLD = 0.5
N_GEN = 20
DEVICE = "cuda"

MODEL_A_LABEL = "TinyLlama"
MODEL_A_CHECKPOINT = "./tinyllama-lora-mcu"
MODEL_A_BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

MODEL_B_LABEL = "Llama-3.2-3B"
MODEL_B_CHECKPOINT = "./llama32-3b-lora-mcu"
MODEL_B_BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

OUTPUT_CSV = "vis/tinyllama_vs_llama32_comparison.csv"
OUTPUT_PLOT = "vis/tinyllama_vs_llama32_comparison.png"

print(f"Selected benchmark slice: {DOMAIN} / {TOPIC}")


In [ ]:
import shlex

compare_cmd = [
    "python",
    "compare_lora_models.py",
    "--repo_path", WILDBENCH_ROOT,
    "--domain", DOMAIN,
    "--topic", TOPIC,
    "--embedder_path", EMBEDDER_PATH,
    "--embedder_label", EMBEDDER_LABEL,
    "--k", str(K),
    "--limit", str(LIMIT),
    "--chunk_size", str(CHUNK_SIZE),
    "--word_coverage_threshold", str(WORD_COVERAGE_THRESHOLD),
    "--n_gen", str(N_GEN),
    "--device", DEVICE,
    "--model_a_label", MODEL_A_LABEL,
    "--model_a_checkpoint", MODEL_A_CHECKPOINT,
    "--model_a_base_model", MODEL_A_BASE_MODEL,
    "--model_b_label", MODEL_B_LABEL,
    "--model_b_checkpoint", MODEL_B_CHECKPOINT,
    "--model_b_base_model", MODEL_B_BASE_MODEL,
    "--output_csv", OUTPUT_CSV,
    "--output_plot", OUTPUT_PLOT,
]

compare_cmd_str = " ".join(shlex.quote(part) for part in compare_cmd)
print(compare_cmd_str)
!{compare_cmd_str}


In [ ]:
from pathlib import Path

csv_path = Path(OUTPUT_CSV)
plot_path = Path(OUTPUT_PLOT)

if not csv_path.exists():
    raise FileNotFoundError(f"Expected CSV was not created: {csv_path}")
if not plot_path.exists():
    raise FileNotFoundError(f"Expected plot was not created: {plot_path}")

print(f"Comparison CSV: {csv_path.resolve()}")
print(f"Comparison plot: {plot_path.resolve()}")
